In [1]:
import cv2

In [2]:
vidname = 'highway1'
vidcap = cv2.VideoCapture(f'phone_films/{vidname}.mov')
success,image = vidcap.read()
count = 0
while success:
  cv2.imwrite(f'frames/{vidname}/frame%d.jpg' % count, image)      
  success,image = vidcap.read()
  print('Read a new frame: ', success)
  count += 100

Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame:  True
Read a new frame

In [ ]:
start = 84000
end = 129000

In [20]:
import torch
import cv2
import numpy as np
from sam3.model_builder import build_sam3_image_model

# SAM3 requires CUDA and is not compatible with CPU-only PyTorch on macOS
# Here's what you can do instead:

device = "cpu"  # macOS doesn't have CUDA

print(f"Torch CUDA available: {torch.cuda.is_available()}")
print(f"Using device: {device}")
print()
print("⚠️ NOTE: SAM3 is designed for CUDA and requires GPU for forward_grounding().")
print()
print("Recommended alternatives for frame segmentation on macOS:")
print("1. Use SAM 1/2 (has better CPU support)")
print("2. Use a lighter segmentation model (e.g., YOLOv8-seg)")
print("3. Extract frames and process them server-side with GPU")
print()
print("For now, here's how to prepare frames for segmentation:")

# Example: prepare a frame for segmentation
if image is not None:
    frame_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    frame_tensor = torch.from_numpy(frame_rgb).float().permute(2, 0, 1).unsqueeze(0)
    
    print(f"Frame shape: {frame_tensor.shape}")
    print(f"Frame dtype: {frame_tensor.dtype}")
    print(f"Frame device: {frame_tensor.device}")
    print()
    print("✓ Frame is ready for processing (would need GPU for SAM3)")


Torch CUDA available: False
Using device: cpu

⚠️ NOTE: SAM3 is designed for CUDA and requires GPU for forward_grounding().

Recommended alternatives for frame segmentation on macOS:
1. Use SAM 1/2 (has better CPU support)
2. Use a lighter segmentation model (e.g., YOLOv8-seg)
3. Extract frames and process them server-side with GPU

For now, here's how to prepare frames for segmentation:


In [21]:
# CPU-Compatible Segmentation Alternative
# Using simple OpenCV-based segmentation that works on macOS

def segment_frame_cpu(frame, method="hsv_range"):
    """
    Segment a frame on CPU using traditional computer vision methods.
    
    Args:
        frame: BGR image from cv2
        method: 'hsv_range' (color-based) or 'background_subtraction'
    
    Returns:
        mask: Binary segmentation mask
    """
    if method == "hsv_range":
        # Example: segment objects in a specific color range (red objects)
        frame_hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        
        # Red range in HSV
        lower_red1 = np.array([0, 100, 100])
        upper_red1 = np.array([10, 255, 255])
        lower_red2 = np.array([170, 100, 100])
        upper_red2 = np.array([180, 255, 255])
        
        mask1 = cv2.inRange(frame_hsv, lower_red1, upper_red1)
        mask2 = cv2.inRange(frame_hsv, lower_red2, upper_red2)
        mask = cv2.bitwise_or(mask1, mask2)
        
    elif method == "background_subtraction":
        # Use MOG2 for motion-based segmentation (good for videos)
        fgbg = cv2.createBackgroundSubtractorMOG2()
        mask = fgbg.apply(frame)
    
    return mask

# Test on current frame
if image is not None:
    mask = segment_frame_cpu(image, method="hsv_range")
    print(f"Segmentation mask shape: {mask.shape}")
    print(f"Mask dtype: {mask.dtype}")
    print(f"Foreground pixels: {np.count_nonzero(mask)}")
    
    # Save example
    cv2.imwrite(f'frames/{vidname}/segmentation_example.jpg', mask)